### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [13]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
docs = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(docs)

In [14]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦'),
 Document(metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct oursel

In [15]:
embeddings = OllamaEmbeddings(model="gemma:2b")
db = FAISS.from_documents(docs, embeddings)
db

#### Query the vector store

In [ ]:
query = "How does the speaker describe outcome of the war?"
docs = db.similarity_search(query)
print(docs[0].page_content)

It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.


#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [18]:
retriever = db.as_retriever()
retriever.invoke(query)
print(docs[0].page_content)

It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.


#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [21]:
ss_score = db.similarity_search_with_score(query)
ss_score

[(Document(id='55c19fc9-9bd4-4be5-b453-d978c606052d', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(3006.5447)),
 (Document(id='fbe9692f-570a-49fa-9bf0-0c9b42080227', metadata={'source': 'speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everythi

#### Passing vectors directly to FAISS
You can also pass vectors directly to FAISS. This is useful if you want to do something with the vectors themselves, or if you want to use a different method of similarity search that FAISS provides. To do this, you can use the similarity_search_by_vector method.

In [22]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[0.31036972999572754,
 -2.3264572620391846,
 -0.5211076736450195,
 2.166687250137329,
 1.4566359519958496,
 0.40595486760139465,
 -1.1208775043487549,
 -0.5313442349433899,
 0.6454050540924072,
 -0.29111233353614807,
 1.8030517101287842,
 0.07514813542366028,
 -1.727645754814148,
 1.2428587675094604,
 -0.6630566716194153,
 -0.7879208326339722,
 2.350451946258545,
 0.7340413928031921,
 -0.005024117883294821,
 0.3062525689601898,
 -0.2488694190979004,
 -0.25921106338500977,
 -0.5210179090499878,
 1.065361738204956,
 -0.38004615902900696,
 -0.415769100189209,
 -0.8328283429145813,
 -1.7052103281021118,
 0.27096763253211975,
 -2.527470827102661,
 -0.039404284209012985,
 -1.416609525680542,
 1.5445667505264282,
 -0.418941855430603,
 0.07597437500953674,
 -0.51800936460495,
 1.306909203529358,
 0.8083475828170776,
 0.732304573059082,
 -0.5631630420684814,
 0.7184777855873108,
 0.0877070501446724,
 0.6858563423156738,
 -2.393122673034668,
 -1.0307228565216064,
 0.40748631954193115,
 0.7501025

In [25]:
docs_by_vector = db.similarity_search_by_vector(embedding_vector)
docs_by_vector

[Document(id='55c19fc9-9bd4-4be5-b453-d978c606052d', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='fbe9692f-570a-49fa-9bf0-0c9b42080227', metadata={'source': 'speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pr

#### Saving and loading FAISS vector stores
FAISS vector stores can be saved and loaded using the save_local and load_local methods.

In [26]:
db.save_local("faiss_index")

In [29]:
new_db = FAISS.load_local(
    "faiss_index", 
    embeddings, 
    allow_dangerous_deserialization=True
)
docs = new_db.similarity_search(query)

In [33]:
docs

[Document(id='55c19fc9-9bd4-4be5-b453-d978c606052d', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='fbe9692f-570a-49fa-9bf0-0c9b42080227', metadata={'source': 'speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pr